In [1]:
import os
import csv
import glob

def process_stentor_results(results_dir, ground_truth_dir):
    # 1. Load Ground Truth Data
    # Structure: { 'a': set(filenames), 'c_0normal': set(filenames), ... }
    ground_truth = {}
    gt_files = glob.glob(os.path.join(ground_truth_dir, "*.txt"))
    
    for gt_path in gt_files:
        category = os.path.basename(gt_path).replace(".txt", "")
        with open(gt_path, 'r') as f:
            # Strip whitespace and filter out empty lines
            filenames = {line.strip() for line in f if line.strip()}
            ground_truth[category] = filenames

    # 2. Process CSV Analysis Results
    # Structure: { category_name: {'yes': count, 'no': count} }
    stats = {}
    csv_files = glob.glob(os.path.join(results_dir, "*_stentor_analysis_results.csv"))

    for csv_path in csv_files:
        # Extract base letter (a, b, c, or d) from filename
        # e.g., "a_stentor_analysis_results.csv" -> "a"
        base_cat = os.path.basename(csv_path).split('_')[0].lower()
        
        # Find all ground truth categories that match this base letter
        # This catches "a" for "a" and "c_0normal", "c_1quick" for "c"
        matched_gt_categories = [cat for cat in ground_truth if cat == base_cat or cat.startswith(f"{base_cat}_")]

        with open(csv_path, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                filename = row['filename'].strip()
                response = row['response'].strip().lower()
                
                # Determine if the response is a 'yes' or 'no'
                is_yes = response.startswith('yes')
                is_no = response.startswith('no')
                
                if not (is_yes or is_no):
                    continue

                # Check this filename against all relevant ground truth variants
                for gt_cat in matched_gt_categories:
                    if filename in ground_truth[gt_cat]:
                        if gt_cat not in stats:
                            stats[gt_cat] = {'yes': 0, 'no': 0}
                        
                        if is_yes:
                            stats[gt_cat]['yes'] += 1
                        else:
                            stats[gt_cat]['no'] += 1

    # 3. Reporting Logic
    print("--- Detailed Reports ---")
    
    # Process variants (like c_0normal) first
    c_variants = sorted([cat for cat in stats if cat.startswith("c_")])
    c_total_yes = 0
    c_total_no = 0

    for var in c_variants:
        y, n = stats[var]['yes'], stats[var]['no']
        print(f"{var}: {y}, {n}")
        c_total_yes += y
        c_total_no += n

    # Print aggregated C
    if c_variants or 'c' in stats:
        # Add any standalone 'c' stats if they exist
        c_total_yes += stats.get('c', {}).get('yes', 0)
        c_total_no += stats.get('c', {}).get('no', 0)
        print(f"\nc: {c_total_yes}, {c_total_no}")
        print("-" * 20)

    # Print others (a, b, d)
    for main_cat in ['a', 'b', 'd']:
        if main_cat in stats:
            y, n = stats[main_cat]['yes'], stats[main_cat]['no']
            print(f"{main_cat}: {y}, {n}")
            
    print("--- Direct Paste  ---")
    
    for var in c_variants:
        y, n = stats[var]['yes'], stats[var]['no']
        print(f"{y}\t{n}")

    print("-" * 20)

    for main_cat in ['a', 'b', 'd']:
        if main_cat in stats:
            y, n = stats[main_cat]['yes'], stats[main_cat]['no']
            print(f"{y}\t{n}")

In [5]:
# --- Execution ---
res_path = "/Users/kesiyun/Desktop/00workspace/bots/API_csvs/235b-instruct"
gt_path = "/Users/kesiyun/Desktop/00workspace/bots/ground_truth/seg"

process_stentor_results(res_path, gt_path)

--- Detailed Reports ---
c_0normal: 50, 1
c_1quick: 7, 2
c_2shortening: 0, 7
c_8dubious: 2, 7
c_9notable: 0, 1

c: 59, 18
--------------------
a: 7, 13
b: 5, 8
d: 4, 20
--- Direct Paste  ---
50	1
7	2
0	7
2	7
0	1
--------------------
7	13
5	8
4	20


In [2]:
# --- Execution ---
res_path = "/Users/kesiyun/Desktop/00workspace/bots/API_csvs/testset_csvs"
gt_path = "/Users/kesiyun/Desktop/00workspace/bots/ground_truth/testset"

process_stentor_results(res_path, gt_path)

--- Detailed Reports ---

c: 18, 11
--------------------
a: 1, 1
b: 1, 2
d: 2, 2
--- Direct Paste  ---
--------------------
1	1
1	2
2	2


In [10]:
# --- Execution ---
res_path = "/Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g_cleaned"
gt_path = "/Users/kesiyun/Desktop/00workspace/bots/ground_truth/seg"

process_stentor_results(res_path, gt_path)

--- Detailed Reports ---
c_0normal: 51, 0
c_1quick: 5, 4
c_2shortening: 3, 4
c_8dubious: 3, 6

c: 62, 14
--------------------
a: 7, 13
b: 3, 10
d: 1, 23
--- Direct Paste  ---
51	0
5	4
3	4
3	6
--------------------
7	13
3	10
1	23


In [8]:
import os
import glob

def clean_csv_bom(input_dir, output_dir=None):
    """
    Remove BOM from all CSV files in a directory.
    
    - input_dir: folder containing CSV files
    - output_dir: where cleaned files go (None = overwrite original)
    """
    if output_dir:
        os.makedirs(output_dir, exist_ok=True)

    csv_files = glob.glob(os.path.join(input_dir, "*.csv"))

    for csv_path in csv_files:
        with open(csv_path, 'r', encoding='utf-8-sig') as f:
            content = f.read()

        # Decide output path
        if output_dir:
            out_path = os.path.join(output_dir, os.path.basename(csv_path))
        else:
            out_path = csv_path  # overwrite original

        with open(out_path, 'w', encoding='utf-8', newline='') as f:
            f.write(content)

        print(f"Cleaned: {csv_path} -> {out_path}")


# ✅ Usage
input_folder = "/Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g"
output_folder = "/Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g_cleaned"

#clean_csv_bom(input_folder, output_folder)
'''
Cleaned: /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g/c_stentor_analysis_results.csv -> /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g_cleaned/c_stentor_analysis_results.csv
Cleaned: /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g/b_stentor_analysis_results.csv -> /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g_cleaned/b_stentor_analysis_results.csv
Cleaned: /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g/a_stentor_analysis_results.csv -> /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g_cleaned/a_stentor_analysis_results.csv
Cleaned: /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g/d_stentor_analysis_results.csv -> /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g_cleaned/d_stentor_analysis_results.csv
'''
pass

Cleaned: /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g/c_stentor_analysis_results.csv -> /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g_cleaned/c_stentor_analysis_results.csv
Cleaned: /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g/b_stentor_analysis_results.csv -> /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g_cleaned/b_stentor_analysis_results.csv
Cleaned: /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g/a_stentor_analysis_results.csv -> /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g_cleaned/a_stentor_analysis_results.csv
Cleaned: /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g/d_stentor_analysis_results.csv -> /Users/kesiyun/Desktop/00workspace/bots/gemini_csvs/6g_cleaned/d_stentor_analysis_results.csv


In [10]:
import os
import csv
import glob


def process_stentor_results_resting(results_dir, ground_truth_dir):
    print(results_dir)
    # 1. Load Ground Truth Data
    # Structure: { 'a': set(filenames), 'c_0normal': set(filenames), ... }
    ground_truth = {}
    gt_files = glob.glob(os.path.join(ground_truth_dir, "*.txt"))
    
    for gt_path in gt_files:
        category = os.path.basename(gt_path).replace(".txt", "")
        with open(gt_path, 'r') as f:
            # Strip whitespace and filter out empty lines
            filenames = {line.strip() for line in f if line.strip()}
            ground_truth[category] = filenames

    # 2. Process CSV Analysis Results
    # Structure: { category_name: {'yes': count, 'no': count} }
    stats = {}
    csv_files = glob.glob(os.path.join(results_dir, "*_stentor_analysis_results.csv"))

    for csv_path in csv_files:
        # Extract base letter (a, b, c, or d) from filename
        # e.g., "a_stentor_analysis_results.csv" -> "a"
        base_cat = os.path.basename(csv_path).split('_')[0].lower()
        
        # Find all ground truth categories that match this base letter
        # This catches "a" for "a" and "c_0normal", "c_1quick" for "c"
        matched_gt_categories = [cat for cat in ground_truth if cat == base_cat or cat.startswith(f"{base_cat}_")]

        with open(csv_path, 'r', encoding='utf-8') as f:
            reader = csv.DictReader(f)
            for row in reader:
                filename = row['filename'].strip()
                response = row['response'].strip().lower()
                
                # Determine if the response is a 'yes' or 'no'
                is_yes = response.startswith('yes')
                is_no = response.startswith('no')
                
                if not (is_yes or is_no):
                    continue

                # Check this filename against all relevant ground truth variants
                for gt_cat in matched_gt_categories:
                    if filename in ground_truth[gt_cat]:
                        if gt_cat not in stats:
                            stats[gt_cat] = {'yes': 0, 'no': 0}
                        
                        if is_yes:
                            stats[gt_cat]['yes'] += 1
                        else:
                            stats[gt_cat]['no'] += 1

    # 3. Reporting Logic
    print("--- Contraction: FP , TN ---")

    # Print others 
    for main_cat in ['n']:
        if main_cat in stats:
            y, n = stats[main_cat]['yes'], stats[main_cat]['no']
            print(f"{main_cat}: {y}, {n}, {n/(n+y)}")
                


In [11]:
# --- Execution ---
res_path = "/Users/kesiyun/Desktop/00workspace/bots/API_csvs/235b-n"
gt_path = "/Users/kesiyun/Desktop/00workspace/bots/ground_truth/resting"
process_stentor_results_resting(res_path, gt_path)

res_path = "/Users/kesiyun/Desktop/00workspace/bots/ubuntu_csvs/32b"
gt_path = "/Users/kesiyun/Desktop/00workspace/bots/ground_truth/resting"
process_stentor_results_resting(res_path, gt_path)

res_path = "/Users/kesiyun/Desktop/00workspace/bots/ubuntu_csvs/8b"
gt_path = "/Users/kesiyun/Desktop/00workspace/bots/ground_truth/resting"
process_stentor_results_resting(res_path, gt_path)

/Users/kesiyun/Desktop/00workspace/bots/API_csvs/235b-n
--- Contraction: FP , TN ---
n: 3, 82, 0.9647058823529412
/Users/kesiyun/Desktop/00workspace/bots/ubuntu_csvs/32b
--- Contraction: FP , TN ---
n: 12, 73, 0.8588235294117647
/Users/kesiyun/Desktop/00workspace/bots/ubuntu_csvs/8b
--- Contraction: FP , TN ---
n: 16, 69, 0.8117647058823529
